#  Regression on House Pricing Dataset: Regularization and Feature Selection
We consider a reduced version of a dataset containing house sale prices for King County, which includes Seattle. It consists in homes sold between May 2014 and May 2015.

Link to the dataset: https://www.kaggle.com/harlfoxem/housesalesprediction

For each house we know 18 house features (e.g., number of bedrooms, number of bathrooms, etc.) plus its price, that is what we would like to predict.

## Insert your ID number ("numero di matricola") below

In [ ]:
# -- put here your ID Number ("numero di matricola")
ID_number = 3 # COMPLETE

In [ ]:
# -- import all packages needed
# -- pandas: software library for data analysis and data manipulation
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Load the data, remove data samples with missing values (NaN) and take a look at them.

In [ ]:
# -- load the data (csv format)
df = pd.read_csv('kc_house_data/kc_house_data_reduced.csv', sep = ',')

# -- remove the data samples with missing values (NaN)
df = df.dropna()

# -- print the column names and the first 5 rows of the dataframe
print(df.columns)
print('\n')
print(df.head())

Separate input and output data. We want to predict the price by using features other than id and date as input.

In [ ]:
data = df.values # removes headers 
m = df.shape[0]

print("Number of samples:", m)

Y = data[:m,2]
# -- explain why we are excluding ids (and date)
X = data[:m,3:]

# -- print shapes
print("X shape: ", X.shape)
print("Y shape: ", Y.shape)

# description of input data X
pd.DataFrame(X, columns=df.columns[3:], dtype=float).describe()

## Data Pre-Processing

We split the data (with $m = 300$ samples) into 2 parts: 
* Training set: data samples used for optimizing the model parameters (we set $m_{train}=200$ samples);
* Test set: data samples for assessing generalization performance (the remaining ones, i.e., $m_{test}=100$ samples).

In [ ]:
# -- Split data into train (200 = 2/3 of samples) and test (the rest)
m_train = int(2/3 * m)
m_test = m - m_train 

# -- print sizes
print("Amount of data for training and deciding parameters:", m_train)
print("Amount of data for test:", m_test)

In [ ]:
# -- let's use built-in function from scikit-learn to split the data
from sklearn.model_selection import train_test_split

# -- look at the documentation: we need to do train_test_split:
# -- first, we split data into train and test
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = m_test, random_state=ID_number)

In [ ]:
# -- let's check if the sizes match
print(f'X_train size: {X_train.shape}, Y_train size: {Y_train.shape}')
print(f'X_test size: {X_test.shape}, Y_test size: {Y_test.shape}')

Let's standardize the data.

In [ ]:
# -- data pre-processing (leave to studs)
from sklearn import preprocessing
scaler = preprocessing.StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# -- let's check if the scaler did his job
print(f'X_train mean: {np.mean(X_train_scaled)}, X_train std: {np.std(X_train_scaled)}')
print(f'X_test mean: {np.mean(X_test_scaled)}, X_test std: {np.std(X_test_scaled)}')

## Least-Squares Solution

Now, let's compute the <u>Least-Squares</u> estimate using **LinearRegression** class in Scikit-learn, and then let's print the corresponding average loss (MSE loss) in training and test data.

Since the average loss could be quite high, we also may want to compute the coefficient of determination $R^2$ and look at $1 - R^{2}$ to have an idea of what the average loss amounts to. In order to compute the coefficient of determination one can use the usual *score($\cdot$)* function.

In [ ]:
# -- Least-Squares
from sklearn import linear_model 
# -- LR the linear regression model
LR = linear_model.LinearRegression()

# -- fit the model on training data
LR.fit(X_train_scaled, Y_train)

# -- obtain predictions on training data
Y_train_predicted = LR.predict(X_train_scaled)

# -- obtain predictions on test data
Y_test_predicted = LR.predict(X_test_scaled)

# -- coefficients from the model (explain horizontal stack)
w_LR = np.hstack((LR.intercept_, LR.coef_))
print('Linear Regressor parameters:\n', w_LR)

# -- average error in training data
loss_train = np.mean((Y_train - Y_train_predicted)**2) # alternatively: loss_train = np.linalg.norm(Y_train - Y_train_predicted)**2 / m_train

# -- average error in test data
loss_test = np.mean((Y_test - Y_test_predicted)**2) # alternatively: loss_test = np.linalg.norm(Y_test - Y_test_predicted)**2 / m_test

print('---')
# -- print average loss in training data and in test data. Explain the high values of parameters (due to very high house prices)
print(f'Average loss in training data: {loss_train} || {loss_train:.3e}')
print(f'Average loss in test data: {loss_test} || {loss_test:.3e}')
print('---')

# -- print 1 - coefficient of determination in training data and in test data
print(f'1 - R² on training data:   {(1 - LR.score(X_train_scaled, Y_train))}')
print(f'1 - R² on test data: {1 - LR.score(X_test_scaled, Y_test)}')

In [ ]:
# -- define the x locations for the groups
ind = np.arange(1, len(LR.coef_) + 1)

# -- define the width of the bars
width = 0.35 

fig, ax = plt.subplots()
rects1 = ax.bar(ind, LR.coef_, width, color='r', label='LR')
ax.legend()
plt.xlabel('Coefficient Idx')
plt.ylabel('Coefficient Value')
plt.title('LR Coefficient')
plt.show()

-- -
# Regularization

## L1 Regularization and Lasso

We use the routine *lasso_path* from *sklearn.linear_regression* to compute the "lasso path", which corresponds to finding the models with lasso for different values of the regularization parameter $\lambda$. You should first fix a grid of possible values for lambda (the variable "lasso_lams"). For each entry of the vector "lasso_lams" we compute the corresponding model (i.e., the $i$-th column of the vector  "lasso_coefs" should contain the coefficients of the linear model computed using lasso_lams[$i$] as regularization parameter). In general, it is recommended to choose the grid appropriately.

<u>Note</u>: the parameter $\lambda$ is called $\alpha$ in the Lasso model from sklearn


In [ ]:
from sklearn.linear_model import lasso_path

# select a grid of possible regularization parameters 
# (be careful how this is chosen, you may have to refine the choice after having seen the results)

lasso_lams = np.logspace(0, 6, num = 100)

# Use the function lasso_path to compute the "lasso path", passing in input the lambda values
# you have specified in lasso_lams
lasso_lams, lasso_coefs, _ = lasso_path(X_train_scaled, Y_train, alphas = lasso_lams) 

We evaluate the sparsity (i.e., number of non-zero coefficients) in the estimated coefficients as a function of the regularization parameter $\lambda$: to this purpose, we compute the number of non-zero entries in the estimated coefficient vector.

In [ ]:
l0_coef_norm = np.zeros(len(lasso_lams),)

for i in range(len(lasso_lams)):
    l0_coef_norm[i] = sum(lasso_coefs[:,i]!=0)


plt.plot(lasso_lams, l0_coef_norm, marker='o', markersize=5)
plt.xlabel('Lambda')
plt.ylabel('Number of non-zero coefficients')
plt.title('Sparsity Degree')
plt.xscale('log')
plt.show()

### Use $k$-fold Cross-Validation to fix the regularization parameter

Now, we need to decide which value of $\lambda$ to use. In order to do this, we are going to use $k$-fold Cross-Validation.

We are going to use the scikit-learn built-in routine *Lasso* (from the *linear_model* package) to compute the lasso coefficients.

We use *KFold* from *sklearn.cross_validation* to split the data (i.e. X_train and Y_train) into the desired number of folds.

Then we pick $\lambda_{opt}$ to be the chosen value for the regularization parameter.

When data is not 'plentiful', we cannot afford to use fresh validation set to select efficiently the best hypothesis.

#### $k$-fold Cross-Validation

Given $m$, we partition the training set into $k$ folds of size $m/k$. For each fold:

* 1 - perform the training on the union of the other $(k-1)$ folds;
* 2 - treat the left-out fold as a *small validation set*, and estimate the error of the learned hypothesis.

$\longrightarrow$ The final estimate of the true error will be the average of the estimated errors through all the folds, as described above.

When $k = m$, we refer to the above procedure as *leave-one out* validation.

Often cross validation is used for model selection.

At the end, the final hypothesis is obtained from training on the entire training set.

In [ ]:
from sklearn.model_selection import KFold

num_folds = 5

kf = KFold(n_splits = num_folds)

# insert code here ...
# init loss and err arrays

for i, lam in enumerate(lasso_lams):
    
    # insert code here...
 
# -- compute the mean => estimate of train and validation losses and errors for each lam
loss_train_lasso_kfold /= num_folds
loss_val_lasso_kfold /= num_folds
err_train_lasso_kfold /= num_folds
err_val_lasso_kfold /= num_folds


# -- choose the regularization parameter that minimizes the validation loss
lasso_lam_opt = lasso_lams[np.argmin(loss_val_lasso_kfold)]
print('Best value of the regularization parameter:', lasso_lam_opt)
print('Min loss: ', np.min(loss_val_lasso_kfold))
# -- logspace(1, 4) = 0.45749
# -- logspace(1, 6) = 0.4525
print('Min (1 - R²) ', np.min(err_val_lasso_kfold))

Let's plot the Cross-Validation estimates of the loss as a function of the regularization parameter.

In [ ]:
plt.xscale('log')
plt.plot(lasso_lams, loss_train_lasso_kfold, color='r', marker='x')
plt.plot(lasso_lams, loss_val_lasso_kfold, color='b', marker='x')
plt.scatter(lasso_lams[np.argmin(loss_val_lasso_kfold)], loss_val_lasso_kfold[np.argmin(loss_val_lasso_kfold)], color='b', marker='o', linewidths=5)
plt.legend(['Train', 'Validation'])
plt.xlabel('Lambda')
plt.ylabel('Loss')
plt.title('Lasso: choice of regularization parameter')
plt.show()

# -- fit lasso model using the optimal value of the reg. parameter lasso_lam_opt found with k-fold cross validation
lasso_kfold = linear_model.Lasso(alpha=lasso_lam_opt)
lasso_kfold.fit(X_train_scaled, Y_train)

print('Total number of coefficients:', (len(lasso_kfold.coef_)))
print('Number of non-zero coefficients:', (sum(lasso_kfold.coef_ != 0)))
print('Best value of regularization parameter:', (lasso_lam_opt))



## Now let's estimate the lasso coefficients using all the training data and the optimal regularization parameter (chosen at previous step)

Note that we use all the training data to learn the final model for the optimal regularization parameter (once we have chosen the parameter, there is no reason to use less data than what we have). We leave out only the test data so that we can later use it to estimate the generalization error.

In [ ]:
# -- estimate Lasso coefficients with all data (X_train_scaled) using the the optimal value lasso_lam_opt of the regularization parameter

# -- define the model
lasso_reg = linear_model.Lasso(alpha=lasso_lam_opt)

# -- fit using the training data
lasso_reg.fit(X_train_scaled, Y_train)

# -- average loss on training data
loss_train_lasso = np.mean((Y_train - lasso_reg.predict(X_train_scaled))**2)

# -- average loss on test data
loss_test_lasso = np.mean((Y_test - lasso_reg.predict(X_test_scaled))**2)

# -- print average loss in training data and in test data
print('Average loss in training data:', loss_train_lasso)
print('Average loss in test data:', loss_test_lasso)

# -- now print 1 - the coefficient of determination on training and on test data to get an idea to what the average
# -- loss corresponds to
print('1 -  R² on training data:  ', 1 - lasso_reg.score(X_train_scaled, Y_train))
print('1 -  R² on test data:', 1 - lasso_reg.score(X_test_scaled, Y_test))

## We now compare the coefficients from the LR model and the Lasso model

Note that we do not plot the bias (that is not a coefficient of the features)

In [ ]:
# -- compare LR and lasso coefficients

# -- define the x locations for the groups
ind = np.arange(1, len(LR.coef_) + 1)

# -- define the width of the bars
width = 0.35 

fig, ax = plt.subplots()
rects1 = ax.bar(ind, LR.coef_, width, color='r')
rects2 = ax.bar(ind + width, lasso_reg.coef_, width, color='y')
ax.legend((rects1[0], rects2[0]), ('LR', 'Lasso'))
plt.xlabel('Coefficient Idx')
plt.ylabel('Coefficient Value')
plt.title('LR and Lasso Coefficient')
plt.show()

## Ridge Regression

### Use Ridge regression with cross-validation

We perform the L2 regularization for different values of the regularization parameter $\alpha$ (called $\lambda$ in class), and use the Scikit-learn function to perform cross-validation (CV).

In Ridge regression for scikit learn, the objective function is:

$$
    ||y - Xw||^2_2 + \alpha * ||w||^2_2
$$

Note: the CV in Scikit-learn is by default a *stratified* CV, that means that data is split into train-validation while maintaining the proportion of different classes in each fold.

In the code below:
- we use RidgeCV() to select the best value of $\alpha$ with a 5-fold CV with L2 penalty;
- we use Ridge() to learn the best model for the best $\alpha$ for ridge regression using the entire training set

Note that RidgeCV() picks some default values of $\alpha$ to try, but we decide to pass the same values used for the Lasso.




In [ ]:
from typing import Any


from sklearn.model_selection import KFold

num_folds = 5

ridge_alphas = np.logspace(0, 6, num = 100)
kf = KFold(n_splits = num_folds)

# insert code here ...
# init loss and err arrays

for i, alpha in enumerate(ridge_alphas):
    # insert code here... 
    # HINT: use linear_model.Ridge(alpha=alpha) instead of linear_model.Lasso()



# -- compute the mean => estimate of train and validation losses and errors for each lam
loss_train_ridge_kfold /= num_folds
loss_val_ridge_kfold /= num_folds
err_train_ridge_kfold /= num_folds
err_val_ridge_kfold /= num_folds


# -- choose the regularization parameter that minimizes the validation loss
ridge_alpha_opt = ridge_alphas[np.argmin(loss_val_ridge_kfold)]
print('Best value of the regularization parameter:', ridge_alpha_opt)
print('Min loss: ', np.min(loss_val_ridge_kfold))
# -- logspace(1, 4) = 0.45749
# -- logspace(1, 6) = 0.4525
print('Min (1 - R²) ', np.min(err_val_ridge_kfold))

## Now let's estimate the ridge coefficients using all the training data and the optimal regularization parameter (chosen at previous step)

Note that we use all the training data to learn the final model for the optimal regularization parameter (once we have chosen the parameter, there is no reason to use less data than what we have). We leave out only the test data so that we can later use it to estimate the generalization error.

In [ ]:
# -- estimate Lasso coefficients with all data (X_train_scaled) using the the optimal value lasso_lam_opt of the regularization parameter

# -- define the model
ridge_reg = linear_model.Ridge(alpha=ridge_alpha_opt)

# -- fit using the training data
ridge_reg.fit(X_train_scaled, Y_train)

# -- average loss on training data
loss_train_ridge = np.mean((Y_train - ridge_reg.predict(X_train_scaled))**2)

# -- average loss on test data
loss_test_ridge = np.mean((Y_test - ridge_reg.predict(X_test_scaled))**2)

# -- print average loss in training data and in test data
print('Average loss in training data:', loss_train_ridge)
print('Average loss in test data:', loss_test_ridge)

# -- now print 1 - the coefficient of determination on training and on test data to get an idea to what the average
# -- loss corresponds to
print('1 -  R² on training data:  ', 1 - ridge_reg.score(X_train_scaled, Y_train))
print('1 -  R² on test data:', 1 - ridge_reg.score(X_test_scaled, Y_test))

## We now compare the coefficient from the LR model, the lasso model, and the ridge model

In [ ]:

# -- define the x locations for the groups
ind = np.arange(1, len(LR.coef_) + 1)
# -- define the width of the bars
width = 0.25

fig, ax = plt.subplots()
rects1 = ax.bar(ind, LR.coef_, width, color='r')
rects2 = ax.bar(ind + width, lasso_reg.coef_, width, color='y')
rects3 = ax.bar(ind + 2*width, ridge_reg.coef_, width, color='b')
ax.legend((rects1[0], rects2[0], rects3[0]), ('LR', 'Lasso', 'Ridge'))
plt.xlabel('Coefficient Idx')
plt.ylabel('Coefficient Value')
plt.title('LR, Lasso, and Ridge Coefficient')
plt.show()

## Now use all data but the test one to learn the best model (using the chosen model class and parameter, if any), and estimate its generalization error on the test data

In [ ]:
# -- compute best LR model on training data
LR = linear_model.LinearRegression()
LR.fit(X_train_scaled, Y_train)

# -- compute and print 1 - coefficient of determination for the best LR model
print("1 -  R² of LR on test data:   ", 1 - LR.score(X_test_scaled, Y_test))

# -- compute best lasso_reg model on training data
lasso_reg = linear_model.Lasso(alpha=lasso_lam_opt)
lasso_reg.fit(X_train_scaled, Y_train)

# -- compute and print 1 - coefficient of determination for the best lasso model
print("1 -  R² of lasso on test data:", 1 - lasso_reg.score(X_test_scaled, Y_test))

# -- compute best ridge model on training data
ridge_final = linear_model.Ridge(alpha = ridge_alpha_opt)
ridge_final.fit(X_train_scaled, Y_train)

# -- compute and print 1 - coefficient of determination for the best ridge model
print("1 -  R² of ridge on test data:", 1 - ridge_final.score(X_test_scaled, Y_test))

## Now we use other data (from the future!) to see which model really works well

In [ ]:
# -- load the new data
df = pd.read_csv('kc_house_data/kc_house_data_future.csv', sep = ',')

# -- remove the data samples with missing values (NaN)
df = df.dropna() 

print(df.info())

data = df.values

m_future = data.shape[0]
print('Number of samples', m_future)

X_future = data[:m_future,3:]
Y_future = data[:m_future,2]

# -- let's scale the new data with the scaler used at the beginning!
X_future_scaled = scaler.transform(X_future)

# -- compute and print 1 - coefficient of determination for the best LR model
print('1 - R² of LR on future data:   ', 1 - LR.score(X_future_scaled, Y_future))

# -- compute and print 1 - coefficient of determination for the best lasso model
print('1 - R² of lasso on future data:', 1 - lasso_reg.score(X_future_scaled, Y_future))

# -- compute and print 1 - coefficient of determination for the best ridge model
print('1 - R² of ridge on future data:', 1 - ridge_final.score(X_future_scaled, Y_future))

Homework: Try running your experiments using a simple validation set instead of k-fold cross-validation, 
or by standardizing the data outside the fold. Observe and discuss how this affects the results. You might also want to try using "future" data as initial dataset and "reduced" data as new, final data